# Week 04 — BBO capstone driver

Round 4. A structural hypothesis rather than a local move: **do these functions respond to coordinate uniformity?** Several test families are built from symmetric expressions, and if any of these eight are, a point with all coordinates equal should stand out.

F2, F3 and F4 get near-diagonal points (all coordinates within 0.002 of each other). F5 through F8 share a common prefix so their responses can be compared against each other at the same coordinates.

This is a cheap hypothesis test: one round, and a clear negative would rule out a whole class of structure.

In [ ]:
%matplotlib inline
import os, sys, warnings
warnings.filterwarnings("ignore")
# Walk up until bbo.py is found, so the notebook runs from anywhere in the repo.
_root = os.getcwd()
while not os.path.exists(os.path.join(_root, "bbo.py")) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
os.chdir(_root); sys.path.insert(0, _root)
import numpy as np
import pandas as pd
import bbo

WEEK = 4
PRIOR = WEEK - 1          # data state this round was proposed from
SEED = 4
OUTDIR = f"outputs/week{WEEK:02d}"; os.makedirs(OUTDIR, exist_ok=True)

# What each function is getting this round, and why.
PLAN = {
    1: 'low-corner probe',
    2: 'diagonal x1≈x2',
    3: 'diagonal, all coords ≈0.176',
    4: 'diagonal, all coords ≈0.567',
    5: 'shared prefix probe',
    6: 'shared prefix probe',
    7: 'shared prefix probe',
    8: 'shared prefix probe',
}
pd.DataFrame([dict(func=f"F{f}", d=bbo.DIMS[f], move=PLAN[f]) for f in bbo.FUNC_IDS])


## 1. Data — the state this round was proposed from

Best point on record per function, truncated to rounds ≤ 3. Nothing below this cell may look at later rounds.


In [ ]:
# The ledger comes FIRST every round: the best point on record, not the latest one.
led = bbo.ledger(up_to=PRIOR)
led["best"] = led["best"].map(lambda v: f"{v:.6g}")
led["x"] = led["x"].map(bbo.submission)
led


## 2. Proposals — near-diagonal and shared-prefix probes

The spread within each vector is the quantity of interest: near zero means the point is on the diagonal.

In [ ]:
proposals = {
    1: np.array([0.051481, 0.059635]),
    2: np.array([0.145789, 0.145667]),
    3: np.array([0.175986, 0.176018, 0.176111]),
    4: np.array([0.567001, 0.567177, 0.567222, 0.5681]),
    5: np.array([0.821489, 0.548961, 0.574899, 0.611175]),
    6: np.array([0.821489, 0.548961, 0.574899, 0.611175, 0.178585]),
    7: np.array([0.821489, 0.489635, 0.578511, 0.59347, 0.178585, 0.525279]),
    8: np.array([0.785496, 0.489658, 0.571328, 0.59347, 0.178585, 0.525279, 0.478596, 0.019573]),
}

pd.DataFrame([dict(func=f"F{fid}",
                   coord_spread=round(float(proposals[fid].max()-proposals[fid].min()), 4),
                   on_diagonal=bool(proposals[fid].max()-proposals[fid].min() < 0.01),
                   submission=bbo.submission(proposals[fid]))
              for fid in bbo.FUNC_IDS])


### Surrogate trust check

Run before reading any acquisition value, not after.


In [ ]:
# Is each surrogate worth listening to? LOO R2 < 0 means it is worse than
# predicting the mean, and any acquisition value built on it is arbitrary.
rows = []
for fid in bbo.FUNC_IDS:
    X, y, _ = bbo.load(fid, up_to=PRIOR)
    r2 = bbo.fit(fid, up_to=PRIOR).loo_r2() if len(y) >= 4 else float("nan")
    rows.append(dict(func=f"F{fid}", n_data=len(y), loo_r2=round(r2, 3),
                     verdict="broken" if r2 < 0 else "usable" if r2 == r2 else "too few points"))
pd.DataFrame(rows)


### Anchor audit


In [ ]:
ANCHOR = {
    1: [0.018957, 0.259878],
    2: [0.098559, 0.954719],
    3: [0.159998, 0.011915, 0.958587],
    4: [0.611147, 0.607958, 0.671974, 0.601141],
    5: [0.014852, 0.297741, 0.718557, 0.219953],
    6: [0.398747, 0.385554, 0.570014, 0.711777, 0.389141],
    7: [0.151858, 0.148558, 0.071547, 0.258484, 0.285157, 0.741141],
    8: [0.159174, 0.118198, 0.137956, 0.716535, 0.781515, 0.543548, 0.279585, 0.258543],
}
# Anchor audit: is each proposal being generated from the best point on record?
# This is the check whose absence cost the campaign most of its final score.
for fid in bbo.FUNC_IDS:
    w = bbo.anchor_check(fid, np.array(ANCHOR[fid], float), up_to=PRIOR)
    print(f"F{fid}: {w if w else 'anchored on best-known point'}")


## 3. Visualise

Best-so-far trajectory per function, truncated to the data available this round.


In [ ]:
import matplotlib
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(15, 6))
for ax, fid in zip(axes.ravel(), bbo.FUNC_IDS):
    try:
        _, y, rounds = bbo.load(fid, up_to=PRIOR)
    except ValueError:
        ax.set_title(f"F{fid}: no data"); continue
    ax.plot(rounds, y, "o", ms=4, alpha=.55)
    ax.plot(rounds, np.maximum.accumulate(y), "-", lw=2)
    ax.set_title(f"F{fid} (d={bbo.DIMS[fid]})", fontsize=9)
    ax.tick_params(labelsize=7); ax.set_xlabel("round", fontsize=8)
fig.suptitle(f"Best so far through round {PRIOR}", fontsize=11)
fig.tight_layout(); fig.savefig(f"{OUTDIR}/trajectories.png", dpi=140)
plt.show()


## 4. Submission strings


In [ ]:
# Portal format: six decimals, dash-separated, one line per function, no labels.
for fid in bbo.FUNC_IDS:
    print(bbo.submission(proposals[fid]))


## 5. After the portal returns each y

Returns recorded below and folded into `bbo.HISTORY` so the next round sees them.


In [ ]:
# Week 4 portal returns - already folded into bbo.HISTORY.
# returned_y = {
#     1: 1.6545027412373418e-187,
#     2: 0.01484628428059747,
#     3: -0.1470980607233212,
#     4: -9.692519234626918,
#     5: 81.79728791398944,
#     6: -0.80217118935894,
#     7: 0.18430174146238856,
#     8: 7.781207717733601,
# }
#
# Diagonal hypothesis: not supported. F4's fully-diagonal point returned -9.69, worse
# than W2's -11.66 only marginally and far off F5's scale. F5 returned 81.80 from the
# shared-prefix point - second highest of the campaign, and again from a probe rather
# than from an optimiser.
#
# for fid, y in returned_y.items():
#     bbo.append_result(fid, proposals[fid], y, rnd=WEEK)
